# 01 – Data collection and cleaning (Limpopo)

**Project:** AI-Based Job Matching and Skills Development in Local Economic Contexts
**Step 1 of the implementation plan.**

This notebook:
1. Loads the ESCO skills, occupations and occupation–skill relations (the *generic layer*).
2. Builds a flat skills vocabulary (preferred + alternative labels) for skill extraction later.
3. Loads the Limpopo job postings you collected into `data/raw/jobs/limpopo_jobs.csv`.
4. Cleans the text and removes duplicates.
5. Tags each posting with its Limpopo district.
6. Saves clean files to `data/processed/` and prints a summary.

Run the cells from top to bottom (Shift + Enter).

In [ ]:
from pathlib import Path
import re
import html
import pandas as pd

# The notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
ESCO_DIR = RAW / "esco"
JOBS_FILE = RAW / "jobs" / "limpopo_jobs.csv"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# While you are testing, keep the example rows. Set to False once you have real postings.
INCLUDE_EXAMPLES = False

print("Project root:", ROOT)
print("ESCO folder found:", ESCO_DIR.exists())
print("Jobs file found:  ", JOBS_FILE.exists())

## 1. Load ESCO (generic layer)

- `skills_en.csv`: every skill, with a preferred label and alternative labels (one per line).
- `occupations_en.csv`: occupations with their ISCO codes (OFO is also built on ISCO, which lets us link them later).
- `occupationSkillRelations_en.csv`: which skills each occupation needs (essential or optional). Used later to generate synthetic candidates.

In [ ]:
skills = pd.read_csv(ESCO_DIR / "skills_en.csv")
occupations = pd.read_csv(ESCO_DIR / "occupations_en.csv")
occ_skill = pd.read_csv(ESCO_DIR / "occupationSkillRelations_en.csv")

print(f"Skills:               {len(skills):,}")
print(f"Occupations:          {len(occupations):,}")
print(f"Occupation–skill links: {len(occ_skill):,}")
skills[["preferredLabel", "altLabels", "skillType"]].head()

## 2. Build the generic skills vocabulary

Each skill has one preferred label and often several alternative labels (e.g. *"use spreadsheets software"* is also *"use Excel"*).
We turn this into one row per label, all pointing to the same skill ID, so the extractor can match any wording.

In [ ]:
def explode_labels(df, id_col="conceptUri"):
    rows = []
    for _, r in df.iterrows():
        pref = str(r["preferredLabel"]).strip()
        rows.append((r[id_col], pref, pref, "preferred"))
        alts = r.get("altLabels")
        if isinstance(alts, str):
            for alt in alts.split("\n"):
                alt = alt.strip()
                if alt and alt.lower() != pref.lower():
                    rows.append((r[id_col], pref, alt, "alternative"))
    out = pd.DataFrame(rows, columns=["skill_id", "skill_name", "label", "label_type"])
    out["label_lower"] = out["label"].str.lower()
    return out.drop_duplicates(subset=["skill_id", "label_lower"])

esco_vocab = explode_labels(skills)
esco_vocab["layer"] = "generic"
print(f"Vocabulary entries: {len(esco_vocab):,} labels for {esco_vocab['skill_id'].nunique():,} skills")
esco_vocab.sample(min(5, len(esco_vocab)), random_state=1)

## 3. Load the Limpopo job postings

Collect postings into `data/raw/jobs/limpopo_jobs.csv` (use the template provided). Required columns:

| Column | What to put in it |
|---|---|
| `job_id` | Your own ID, e.g. `DPSA-0001`, `PNET-0001` |
| `title` | Job title as advertised |
| `description` | Full advert text: duties, requirements, qualifications |
| `employer` | Company or department |
| `town` | Town as written in the advert, e.g. *Polokwane*, *Tzaneen* |
| `date_posted` | YYYY-MM-DD |
| `source` | DPSA, ESSA, SAYouth, PNet, Careers24, CareerJunction, Gumtree, Other |
| `url` | Link to the advert (for your records) |

**Target:** 300–500 postings, spread across sources and districts, including some informal and lower-skill jobs.
Do not record personal details of individuals (e.g. contact names or phone numbers in Gumtree ads) — POPIA.

In [ ]:
REQUIRED = ["job_id", "title", "description", "employer", "town", "date_posted", "source", "url"]

jobs_raw = pd.read_csv(JOBS_FILE, dtype=str).fillna("")
missing = [c for c in REQUIRED if c not in jobs_raw.columns]
if missing:
    raise ValueError(f"Missing columns in {JOBS_FILE.name}: {missing}")

is_example = jobs_raw["source"].str.upper().str.startswith("EXAMPLE")
if not INCLUDE_EXAMPLES:
    jobs_raw = jobs_raw[~is_example]
print(f"Loaded {len(jobs_raw)} postings ({is_example.sum()} example rows in file)")
jobs_raw.head()

## 4. Clean the text and remove duplicates

- Strip HTML tags and entities, bullet symbols and extra whitespace.
- Standardise town and source names.
- Drop postings with no description.
- Remove duplicates: the same job is often posted on several sites, so we match on title + employer + town, and also on identical description text.

In [ ]:
def clean_text(t):
    t = html.unescape(str(t))
    t = re.sub(r"<[^>]+>", " ", t)                 # HTML tags
    t = re.sub(r"[•●▪■◦•]", " ", t)       # bullet symbols
    t = re.sub(r"\s+", " ", t)                       # whitespace
    return t.strip()

jobs = jobs_raw.copy()
for col in ["title", "description", "employer", "town", "source"]:
    jobs[col] = jobs[col].map(clean_text)

jobs["town"] = jobs["town"].str.title()
jobs["date_posted"] = pd.to_datetime(jobs["date_posted"], errors="coerce")

before = len(jobs)
jobs = jobs[jobs["description"].str.len() > 30]
empty_dropped = before - len(jobs)

key = (jobs["title"].str.lower() + "|" + jobs["employer"].str.lower() + "|" + jobs["town"].str.lower())
jobs = jobs[~key.duplicated()]
jobs = jobs[~jobs["description"].str.lower().duplicated()]
dupes_dropped = before - empty_dropped - len(jobs)

print(f"Dropped {empty_dropped} with no/very short description, {dupes_dropped} duplicates")
print(f"Remaining postings: {len(jobs)}")

## 5. Tag each posting with its Limpopo district

Limpopo has five district municipalities. The lookup below covers the main towns; add any new towns you meet.
Postings whose town is not in the lookup are marked **Unknown** and listed so you can add them.

In [ ]:
DISTRICT_TOWNS = {
    "Capricorn":  ["Capricorn", "Polokwane", "Pietersburg", "Seshego", "Mankweng", "Lebowakgomo", "Moletji",
                   "Senwabarwana", "Alldays", "Mogwadi", "Dendron", "Morebeng", "Soekmekaar", "Zebediela",
                   "Turfloop", "Botlokwa", "Matlala", "Aganang", "Buffelshoek"],
    "Vhembe":     ["Vhembe", "Thohoyandou", "Makhado", "Louis Trichardt", "Musina", "Malamulele", "Elim",
                   "Sibasa", "Mutale", "Vuwani", "Shayandima", "Siloam", "Tshipise", "Nzhelele"],
    "Mopani":     ["Mopani", "Tzaneen", "Phalaborwa", "Giyani", "Modjadjiskloof", "Letsitele", "Hoedspruit",
                   "Namakgale", "Lenyenye", "Haenertsburg", "Gravelotte", "Nkowankowa", "Kgapane",
                   "Sekororo", "Maruleng"],
    "Sekhukhune": ["Sekhukhune", "Jane Furse", "Groblersdal", "Marble Hall", "Burgersfort", "Steelpoort",
                   "Ga-Nkwana", "Ohrigstad", "Driekop", "Motetema", "Dennilton", "Glen Cowie", "Mecklenburg",
                   "Atok", "Matlerekeng"],
    "Waterberg":  ["Waterberg", "Mogalakwena", "Mokopane", "Lephalale", "Ellisras", "Modimolle", "Bela-Bela",
                   "Thabazimbi", "Mookgophong", "Marapong", "Vaalwater", "Northam", "Witpoort", "Bakenberg",
                   "Thabaleshoba", "Masebe", "Nylsvlei"],
}
TOWN_TO_DISTRICT = {t.lower(): d for d, towns in DISTRICT_TOWNS.items() for t in towns}

def to_district(town):
    t = town.lower()
    if t in TOWN_TO_DISTRICT:
        return TOWN_TO_DISTRICT[t]
    for name, d in TOWN_TO_DISTRICT.items():      # e.g. "Polokwane Central"
        if name in t:
            return d
    return "Unknown"

jobs["district"] = jobs["town"].map(to_district)
unknown = sorted(jobs.loc[jobs["district"] == "Unknown", "town"].unique())
print("District counts:")
print(jobs["district"].value_counts().to_string())
if unknown:
    print("\nTowns to add to the lookup:", unknown)

## 6. Summary

Check the balance of your dataset before moving on. In the write-up, report these numbers in your *Data* section.

In [ ]:
jobs["desc_words"] = jobs["description"].str.split().str.len()

print(f"Total postings: {len(jobs)}\n")
print("By source:")
print(jobs["source"].value_counts().to_string(), "\n")
print("By district:")
print(jobs["district"].value_counts().to_string(), "\n")
print("Description length (words):")
print(jobs["desc_words"].describe().round(0).to_string())
if jobs["date_posted"].notna().any():
    print(f"\nDate range: {jobs['date_posted'].min().date()} to {jobs['date_posted'].max().date()}")

## 7. Save the clean outputs

In [ ]:
jobs.to_csv(PROCESSED / "jobs_clean.csv", index=False)
esco_vocab.to_csv(PROCESSED / "vocab_generic.csv", index=False)
occupations.to_csv(PROCESSED / "esco_occupations.csv", index=False)
occ_skill.to_csv(PROCESSED / "esco_occupation_skills.csv", index=False)

for f in sorted(PROCESSED.glob("*.csv")):
    print(f"{f.name:32s} {f.stat().st_size/1024:8.0f} KB")

**Next notebook:** `02_local_vocabulary.ipynb` — builds the local layer (OFO codes, SA credentials such as matric, NC(V), N4–N6, trade test, learnerships) and the credential equivalence table.